# Data Mining Assignment: Naive Bayes vs Logistic Regression

In [ ]:
# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)


In [ ]:
# load the csv and preview it
df = pd.read_csv("diabetes.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
# check column types, missing values, and summary stats
df.info()
df.describe()


In [ ]:
# check class balance and plot it
print(df["Outcome"].value_counts())
sns.countplot(x="Outcome", data=df)
plt.title("Class Distribution (0 = No Diabetes, 1 = Diabetes)")
plt.show()


In [ ]:
# columns where 0 is not a realistic value
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print((df[zero_as_missing] == 0).sum())


In [ ]:
# replace those 0s with NaN, then fill with the column median
df_clean = df.copy()
df_clean[zero_as_missing] = df_clean[zero_as_missing].replace(0, np.nan)
df_clean[zero_as_missing] = df_clean[zero_as_missing].fillna(df_clean[zero_as_missing].median())
df_clean.describe()


In [ ]:
# correlation heatmap between features
plt.figure(figsize=(9, 7))
sns.heatmap(df_clean.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()


In [ ]:
# separate features and target, then split into train and test sets
X = df_clean.drop(columns=["Outcome"])
y = df_clean["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# scale features so they are on a similar range
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape, " Test shape:", X_test_scaled.shape)


## 5. Model 1 — Naive Bayes

Naive Bayes assumes features are independent of each other given the class.


In [ ]:
# train the model and predict on the test set
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

y_pred_nb = nb_model.predict(X_test_scaled)
y_proba_nb = nb_model.predict_proba(X_test_scaled)[:, 1]


In [ ]:
# print evaluation metrics
print("Accuracy: ", accuracy_score(y_test, y_pred_nb))
print("Precision:", precision_score(y_test, y_pred_nb))
print("Recall:   ", recall_score(y_test, y_pred_nb))
print("F1 score: ", f1_score(y_test, y_pred_nb))
print()
print(classification_report(y_test, y_pred_nb))


In [ ]:
# confusion matrix as a heatmap
cm_nb = confusion_matrix(y_test, y_pred_nb)
sns.heatmap(cm_nb, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Diabetes", "Diabetes"],
            yticklabels=["No Diabetes", "Diabetes"])
plt.title("Naive Bayes — Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()


## 6. Model 2 — Logistic Regression

In [ ]:
# train the model and predict on the test set
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)
y_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]


In [ ]:
# print evaluation metrics
print("Accuracy: ", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:   ", recall_score(y_test, y_pred_lr))
print("F1 score: ", f1_score(y_test, y_pred_lr))
print()
print(classification_report(y_test, y_pred_lr))


In [ ]:
# confusion matrix as a heatmap
cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt="d", cmap="Greens",
            xticklabels=["No Diabetes", "Diabetes"],
            yticklabels=["No Diabetes", "Diabetes"])
plt.title("Logistic Regression — Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()


In [ ]:
# see which features influence the prediction most
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lr_model.coef_[0]
}).sort_values("Coefficient", key=abs, ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=coef_df, x="Coefficient", y="Feature", palette="viridis")
plt.title("Logistic Regression Coefficients (Feature Influence)")
plt.axvline(0, color="black", linewidth=0.8)
plt.show()

coef_df


## 7. Model Comparison

In [ ]:
# put both models' scores into one table
results = pd.DataFrame({
    "Model": ["Naive Bayes", "Logistic Regression"],
    "Accuracy": [accuracy_score(y_test, y_pred_nb), accuracy_score(y_test, y_pred_lr)],
    "Precision": [precision_score(y_test, y_pred_nb), precision_score(y_test, y_pred_lr)],
    "Recall": [recall_score(y_test, y_pred_nb), recall_score(y_test, y_pred_lr)],
    "F1 Score": [f1_score(y_test, y_pred_nb), f1_score(y_test, y_pred_lr)],
    "ROC AUC": [roc_auc_score(y_test, y_proba_nb), roc_auc_score(y_test, y_proba_lr)],
})
results


In [ ]:
# bar chart comparing the two models
results.set_index("Model")[["Accuracy", "Precision", "Recall", "F1 Score"]].plot(
    kind="bar", figsize=(9, 5), rot=0
)
plt.title("Naive Bayes vs Logistic Regression — Metric Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.legend(loc="lower right")
plt.show()


In [ ]:
# ROC curves for both models
fpr_nb, tpr_nb, _ = roc_curve(y_test, y_proba_nb)
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)

plt.figure(figsize=(7, 6))
plt.plot(fpr_nb, tpr_nb, label=f"Naive Bayes (AUC = {roc_auc_score(y_test, y_proba_nb):.3f})")
plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {roc_auc_score(y_test, y_proba_lr):.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()


## 8. Conclusion

- Both models were trained on the same data and tested on the same held-out set.
- The table and charts above compare Accuracy, Precision, Recall, F1, and ROC AUC.
- Naive Bayes is simple and fast, but assumes features are independent (not fully true here).
- Logistic Regression can handle correlated features better and gives interpretable coefficients.
